# Discrete utility transform 

    JMA 29 Aug 2026

This notebook compares the risk premium for a normally distributed gamble, as the discretization level is reduced. 

The following analysis shows that using maximum expected utility, computation in the risk averse case always has a positive risk premium, regardless of discretization level.  In fact the discretization tends to overestimate the risk premium at coarse discretizations. 

Risk aversion with MEU works by applying a concave utility function to each prospect (random variable), taking expectation, then applying the inverse utility function to obtain a certain equivalent for the gamble. Let's start with this exponential "delta property" utility function

$u(x) = a - b*\exp(-x/\text{risk\_tolerance})$

where for convenience a and b are set to that the utility function range is 0,1 over the range of the random variable. Granted that the binary approximation is poor - by a factor of 5, using a reasonable number of discrete bins is an obvious precaution, and quickly gives accurate values. 

The analytic solution consists of substituting u(x) into a normal PDF, which looks something like a Gamble distribution of the form

$f(x) = \exp(-\exp(x))$


In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

DEBUG = 0

In [ ]:
# Functions for an exponential utility over x_min, X_MAX, with r - risk odds

def fit_exp(x_min, x_max, r):
    r_min = pow(r, -x_min)
    r_max = pow(r, -x_max)
    b = 1/(r_min - r_max)
    a = r_min * b
    return a, b

def delta_util_pts(x, xm, xM, r):
    if abs(r -1) < 1e-5:
        b = 1/(xm - xM)
        u = b*(xm - x) 
    else:
        a,b = fit_exp(xm, xM, r)
        if DEBUG: print(f'{a:.3f}, {b:.3f}')
        u =  a - b*np.exp(-x * np.log(r))
    return u

def inverse_delta_util(u, xm, xM, r):
    if abs(r -1) < 1e-5:
        x = xm + u *(xM - xm)
    else:
        a,b = fit_exp(xm, xM, r)
        x = - np.log((a - u)/b)/np.log(r)
    return x


In [ ]:
# Transform a discrete normal distribution by an exponential utility

def discrete_comparision(no_bins = 2, mean = MEAN, std = STD, x_min=X_MIN, x_max=X_MAX):
    """
    Compute a normal distribution and discretize it into bins.
    
    Parameters:
    - no_bins: Number of bins for discretization
    - mean: Mean of the normal distribution
    - std: Standard deviation of the normal distribution
    
    Returns:
    - bin_probs: Probability mass in each bin
    - v_bin_centers: center values for untransformed histogram
    - v_bin_edges: histogram plotting edges
    - u_bin_centers: center values for transformed histogram
    - u_bin_edges: histogram plotting edges
    - u_bin_widths: transformed bin widths

    """
    # Create bin edges for values (untransformed normal)
    v_bin_edges = np.linspace(x_min, x_max, no_bins + 1)

    # Compute probability in each bin using the CDF
    normal_dist = stats.norm(mean, std)
    bin_probs = np.diff(normal_dist.cdf(v_bin_edges))
    v_bin_centers = (v_bin_edges[:-1] + v_bin_edges[1:]) / 2
        
    # Transform - linear into utility space
    u_bin_centers = delta_util_pts(v_bin_centers, x_min, x_max, risk_odds)
    u_bin_edges = delta_util_pts(v_bin_edges, x_min, x_max, risk_odds)
    u_bin_widths = np.diff(u_bin_edges)
    return bin_probs, v_bin_centers, v_bin_edges, u_bin_centers, u_bin_edges, u_bin_widths

# Compute expected utilities

def expectation(p, u):
    return (p * u).sum()

# Risk premium as bin count varies

def lotteries(number_bins, x_min=X_MIN, x_max=X_MAX):
    bin_probs, v_bin_centers, v_bin_edges, u_bin_centers, u_bin_edges, u_bin_widths =\
    discrete_comparision(number_bins, mean = MEAN, std = STD)
    # Compare a binary lottery with a continuous, risk averse
    expected_value = expectation(bin_probs, v_bin_centers)
    expected_utility = expectation(bin_probs, u_bin_centers)

    certain_equivalent = inverse_delta_util(expected_utility, x_min, x_max, risk_odds)
    if DEBUG: print(expected_value, expected_utility, certain_equivalent)
    risk_premium = expected_value-certain_equivalent
    print(f'{number_bins} - expected_value, risk premium:  {expected_value:.3f} {risk_premium:.3f}')
    return risk_premium

In [ ]:
# Parameters

RISK_TOLERANCE = 2
risk_odds = np.exp(1/RISK_TOLERANCE)
print(f'risk odds: {risk_odds:.3f}')
X_MIN, X_MAX = 0, 10.0
MEAN = 5
STD = 1.0
NO_BINS = 2

In [ ]:
# Test forward and backward utility transforms 
y = delta_util_pts(np.array([-3,-1,0,1,3]), -3,3,1)
print(y)
x1 = inverse_delta_util(y, -3,3,1 )
print(x1)

In [ ]:
# Test delta utility 

x = np.linspace(X_MIN, X_MAX, 100)
u = delta_util_pts(x, X_MIN, X_MAX, risk_odds)
x1 = inverse_delta_util(u, X_MIN, X_MAX, risk_odds)

plt.plot(x, u)
if DEBUG: plt.plot(x, x1, ':')
plt.xlabel("x")
plt.ylabel("Utility")
plt.title("Exponential Utility Function")
plt.grid(True)
plt.show()

In [ ]:
# Plot untransformed value distributions versus utility distributions
bin_probs, v_bin_centers, v_bin_edges, u_bin_centers, u_bin_edges, u_bin_widths =\
discrete_comparision(NO_BINS, mean = MEAN, std = STD)
v_bin_width = v_bin_edges[1] - v_bin_edges[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot in value space
ax1.bar(v_bin_centers, bin_probs, width=v_bin_width, alpha=0.5,
            edgecolor='black', label='Discrete PDF')
for x_center, p in zip(v_bin_centers, bin_probs):
    ax1.annotate(f'{x_center:.2f}', (x_center, 0), xytext=(0, 6),
                textcoords='offset points', ha='center', va='bottom', fontsize=12)
ax1.set_xlabel('Random variable')
ax1.set_ylabel('Probability Density')
ax1.set_title('Value Discrete PDF')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Plot in utility space
ax2.bar(u_bin_edges[:-1], bin_probs, width=u_bin_widths, align='edge', alpha=0.2, color= 'grey',
            edgecolor='black', label='Discrete PDF')

# Keep the x-axis tick marks, and annotate the transformed centers above the axis
for x_center, p in zip(u_bin_centers, bin_probs):
    ax2.annotate(f'{x_center:.2f}', (x_center, 0), xytext=(0, 6),
                textcoords='offset points', ha='center', va='bottom', fontsize=12)

ax2.set_xlabel('Utility')
ax2.set_title('Utility Discrete PDF')
plt.tight_layout()
plt.show()

In [ ]:
# Compare this with a 2 branch tree
eu = delta_util_pts(2.50, X_MIN, X_MAX, risk_odds)/2 + delta_util_pts(7.50, X_MIN, X_MAX, risk_odds)/2 
5 - inverse_delta_util(eu, X_MIN, X_MAX, risk_odds)

In [ ]:
# Iterate over different bin counts, to approximate risk premium
ind = []
rp = []
for b in range(2,21):
    rp.append(lotteries(b))
    ind.append(b)

In [ ]:
# PLot risk premium as discretication increases. 

plt.plot(ind, rp)
plt.xlabel("Number of Bins")
plt.ylabel("Risk Premium")
plt.title("Risk Premium Convergence")
plt.grid(True)
plt.ylim(0, 1.1 * max(rp))
plt.xticks(range(2,21))
plt.show()


(c) John Mark Agosta